In [1]:
from langgraph.graph import StateGraph , START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
#creating state 
class BatsmanState(TypedDict):

    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundarypercent: float
    summary: str
    

In [24]:
def calculate_sr(state: BatsmanState) -> BatsmanState :
    sr= (state['runs']/state['balls'])*100

    #update the state

    return {'sr' : sr}

In [16]:
def calculate_bpb(state: BatsmanState) :

   bpb =  state['balls']/(state['fours'] + state['sixes'])

   return {'bpb': bpb}

In [25]:
def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = (((state['fours']*4) + (state['sixes']*6))/state['runs'])*100

    return {'boundarypercent': boundary_percent }

In [26]:
def summary(state: BatsmanState):

    summary=f"""
    Strike Rate - {state['sr']}\n
    Balls per Boundary - {state['bpb']}\n
    Boundary Percentage - {state['boundarypercent']}\n
"""
    return {'summary': summary}

In [27]:
#define state 
graph = StateGraph(BatsmanState)
#adding node
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)
#adding edges
graph.add_edge(START , 'calculate_sr')
graph.add_edge(START , 'calculate_bpb')
graph.add_edge(START , 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

#graph compile 
workflow = graph.compile()

#execution

In [28]:
#execution
initial_state={
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4,
}

workflow.invoke(initial_state)

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundarypercent': 48.0,
 'summary': '\n    Strike Rate - 200.0\n\n    Balls per Boundary - 5.0\n\n    Boundary Percentage - 48.0\n\n'}